#Setup Script

In [ ]:
#https://www.tbi.univie.ac.at/RNA/packages/source/ViennaRNA-2.3.0cuda.tar.gz
#Help: http://www0.cs.ucl.ac.uk/staff/ucacbbl/rnafold/

import re
import sys
import os
from os import fsync
import glob
import pickle
import pandas as pd
import numpy as np
import random
import torch



import subprocess

# background sampler: per-core CPU%, and per-thread state/core assignment for RNAfold specifically
monitor = subprocess.Popen(
    ["bash", "-c",
     "while true; do date +%T; mpstat -P ALL 1 1 2>/dev/null; "
     "ps -C RNAfold -L -o pid,tid,pcpu,psr,stat,comm; echo ---; sleep 1; done"],
    stdout=open("cpu_log.txt", "w"), stderr=subprocess.STDOUT)

!RNA_CPU_THREADS=8 ./RNAfold < stress_400.fa > /dev/null

monitor.terminate()





#Get a normal version of RNAFold
!pip -q install viennarna==2.6.4
import RNA

#Get missing NCVV samples
!git clone https://github.com/nvidia/cuda-samples

#Check CPU
print("CPU info:")
!lscpu
print("%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%1")

#Check GPU
print("GPU info:")
!which nvcc
!nvcc --version
!nvidia-smi
gpu_sms = torch.cuda.get_device_capability()
gpu_sms = gpu_sms[0]*10 + gpu_sms[1]
gpu_sms = str(gpu_sms)
print("Device SMS = ", gpu_sms)
!echo {gpu_sms}
assert gpu_sms.isdigit() and len(gpu_sms) >= 2

total_mem = torch.cuda.get_device_properties(torch.cuda.current_device()).total_memory
print("Total GPU memory = ", total_mem)
print("%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%5")

#Commented out the tarball method
#!wget https://www.tbi.univie.ac.at/RNA/packages/source/ViennaRNA-2.3.0cuda.tar.gz
#!tar xzf ViennaRNA-2.3.0cuda.tar.gz

#Pull from my GitHub instead
#!git clone https://github.com/LukeTheGeneWriter/CUDA_RNAFold.git
#!git clone -b CUDA_Graphs_Branch https://github.com/LukeTheGeneWriter/CUDA_RNAFold.git
#!git clone -b GPU_Energy_Precompute_Branch https://github.com/LukeTheGeneWriter/CUDA_RNAFold.git
#!git clone -b Heterogenous_Computing_GPU_Energy_Precompute_CUDA_Graphs https://github.com/LukeTheGeneWriter/CUDA_RNAFold.git
#!git clone -b Langdon_Indexing_Logic_Het_Energy_Precompute_CUDA_Graphs https://github.com/LukeTheGeneWriter/CUDA_RNAFold.git
!git clone -b Profiled_Langdon_Energy_Het_Graphs https://github.com/LukeTheGeneWriter/CUDA_RNAFold.git


#Go into the CUDA_RNAFold directory
%cd CUDA_RNAFold
!cp -r /content/cuda-samples/Common/* /usr/local/cuda/src

#Get build dependencies
!apt-get -qq update
!apt-get -qq install -y check texinfo help2man gengetopt flex sysstat #Added flex dependency during heterogenous compute build and sysstat to check on
!which makeinfo help2man gengetopt || (echo "MISSING BUILD DEPS" && exit 1)

!autoupdate
!chmod +x configure
!./configure NVCC_PATH=/usr/local/cuda/bin NVCC_SAMPLES=/content/cuda-samples/Common NVCC_GENCODE="-arch=native" --prefix=/home/ViennaRNA
#!./configure NVCC_PATH=/usr/local/cuda/bin NVCC_SAMPLES=/content/cuda-samples/Common CUDA_SMS="{gpu_sms}" --prefix=/home/ViennaRNA


#!./configure NVCC_PATH=/usr/local/cuda/bin NVCC_SAMPLES=/content/cuda-samples/Common CUDA_SMS="89" --prefix=/home/ViennaRNA
#!./configure NVCC_PATH=/usr/local/cuda/bin NVCC_SAMPLES=/content/cuda-samples/Common CUDA_SMS="75" --prefix=/home/ViennaRNA #"75 86 89" #NVCCFLAGS=-m64 -Xcompiler -Wall -Xcompiler -fno-strict-aliasing ${NVCC_GENCODE} -I${NVCC_SAMPLES}"""
!./config.status --config

#Quick fix to makefile redo rule
!find . \( -name 'Makefile.am' -o -name '*.m4' \) -exec touch {} \;
!sleep 2
!touch aclocal.m4 configure config.h.in
!find . -name 'Makefile.in' -exec touch {} \;
!sleep 2
!find . -name 'Makefile' -exec touch {} \; 2>/dev/null

#!./configure --with-swig --disable-check-python3 NVCC_PATH=/usr/local/cuda/bin NVCC_SAMPLES=/usr/local/cuda/samples CUDA_SMS="75" --enable-sse #CUDA_SMS="35 50 61" #CUDA_SMS="50" --enable-sse
#!./make #--prefix=/home/ViennaRNA
#!make V=1 && make install
#!./make NVCC_FLAGS="-m64 -Xcompiler -Wall -Xcompiler -fno-strict-aliasing ${NVCC_GENCODE} -I${NVCC_SAMPLES}"
!make V=1 && make install
%cd ..


/bin/bash: line 1: stress_400.fa: No such file or directory
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 92.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Cloning into 'cuda-samples'...
remote: Enumerating objects: 31872, done.
remote: Counting objects: 100% (7352/7352), done.
remote: Compressing objects: 100% (441/441), done.
remote: Total 31872 (delta 6966), reused 6911 (delta 6911), pack-reused 24520 (from 2)
Receiving objects: 100% (31872/31872), 137.28 MiB | 20.17 MiB/s, done.
Resolving deltas: 100% (27705/27705), done.
Updating files: 100% (2022/2022), done.
CPU info:
Architecture:                x86_64
  CPU op-mode(s):            32-bit, 64-bit
  Address sizes:             46 bits physical, 48 bits virtual
  Byte Order:                Little Endian
CPU(s):                      12
  On-line CPU(s) list:       0-11
Vendor ID:                   GenuineIntel
  Mode

#Create random RNAs to test on

In [ ]:
alphabet = ['U','T', 'G', 'C']
num_RNAs = 400
length = 5601
def random_RNA(alphabet,length):
  r_rna = ''
  for i in range(length):
    r_rna += random.choice(alphabet)
  return r_rna

def random_aaseq(length):
  aas = ['S', 'L', 'C', 'W', 'E', 'D', 'P', 'V', 'N', 'M', 'K', 'Y', 'I', 'Q', 'F', 'R', 'T', 'A', 'G', 'H'] #Removed "*"
  ranlen = length - 2
  aa = 'M'
  for i in range(0,ranlen):
    aa += random.choice(aas)
  aa += '*'
  return aa


def random_codon_stream(aaseq):
  '''Take in an aa seq and output a vector of codons that capture the combination space'''
  aaCodonVecs = {
    'S': ['TCT', 'TCC', 'TCA', 'TCG', 'AGT', 'AGC'],
    'L': ['TTA', 'TTG', 'CTT', 'CTC', 'CTA', 'CTG'],
    'C': ['TGT', 'TGC'],
    'W': ['TGG'],
    'E': ['GAA', 'GAG'],
    'D': ['GAT', 'GAC'],
    'P': ['CCT', 'CCC', 'CCA', 'CCG'],
    'V': ['GTT', 'GTC', 'GTA', 'GTG'],
    'N': ['AAT', 'AAC'],
    'M': ['ATG'],
    'K': ['AAA', 'AAG'],
    'Y': ['TAT', 'TAC'],
    'I': ['ATT', 'ATC', 'ATA'],
    'Q': ['CAA', 'CAG'],
    'F': ['TTT', 'TTC'],
    'R': ['CGT', 'CGC', 'CGA', 'CGG', 'AGA', 'AGG'],
    'T': ['ACT', 'ACC', 'ACA', 'ACG'],
    '*': ['TAA', 'TAG', 'TGA'],
    'A': ['GCT', 'GCC', 'GCA', 'GCG'],
    'G': ['GGT', 'GGC', 'GGA', 'GGG'],
    'H': ['CAT', 'CAC']
  }
  codonvec=[]
  for aa in aaseq.upper():
    if aa not in aaCodonVecs.keys():
      raise NotImplementedError
    else:
      codonvec.append(aaCodonVecs[aa])
  codstream = ""
  for c in codonvec:
    assert isinstance(c, list)
    assrt = random.choice(c)
    assert isinstance(assrt, str)
    codstream += assrt
  return codstream


def generate_random_codon_perms(RNAlen, num_RNAs):
  if RNAlen % 3 != 0:
    raise ValueError("Codon combinations can only be made on RNA lengths divisible by 3")
  else:
    r3 = int(RNAlen / 3)
    polyaa = random_aaseq(r3)
    assert len(polyaa) == r3
    cod_perms = []
    for i in range(0, num_RNAs):
      cod_perms.append(random_codon_stream(polyaa))
    assert len(cod_perms) == num_RNAs

    return cod_perms

#Use this command for random RNAs
#RNA_list = [random_RNA(alphabet,length) for i in range(num_RNAs)]

#Use this command for synonymous RNAs
RNA_list = generate_random_codon_perms(length, num_RNAs)

def make_in_fasta(RNA_list, dest):
  '''turn the RNA seq list into a fasta for input'''
  label1 = "> RNA "
  label2= " for RNAFold GPU testing"
  i = 0
  with open(dest, 'wt') as wfile:
    for RNA in RNA_list:
      head = label1 + str(i) + label2
      wfile.write(head)
      wfile.write("\n")
      wfile.write(RNA)
      wfile.write("\n")
      i += 1
  return

def read_fasta(fastapath, only_last=False):
  if only_last:
    with open(fastapath, 'r') as f:
      lines = f.readlines()
    for line in lines[-1:]:
      print(line)
  else:
    with open(fastapath, 'r') as f:
      lines = f.readlines()
    for line in lines:
      print(line)

dst = "./RNA_list.fasta"
make_in_fasta(RNA_list, dst)
read_fasta(dst, only_last=True)

ATGCGCTATAATCTGACTAATAACACTGTCGGGTATTTTAGACACACAGGGAAGTATGATATGGATTGTATTGCATGGTTTCTGTCAAAATTTGAAGAGACAGGTCACGTAGATTATGACACCTTGATGTACCGCTGGCACACTGGCAAATACATAACAAACGAAGAGCGTAACTGGTGGCTGAATTGTCAGGGAACGGAGGCAGGGTCCCGCACAGACAACTACCAATACTTCCACTACCGTCGAGTAACCAACCGCCAGGATAGGGGAGACTGTGGCCAGATGGCCCCGGTTTACTGGCAAGATAATACTTCCTGGGAAACCGACCCTATGTGTAATGAGCCAGTTTGTCATGTATTCGCGTGGTGGGCCTGGATCTGGCACCTCTGTGACGACTATGCGGAAATTGGAGCGTGTCCATGGGGAGAGATATGCTATGTGCATAACTATCCGTGGAAGGTCGAGAGTGTTATCTGTGCTGGGTATATAAACATGTGGACCATGTGGCGTCCAGGGCATTATGTTCTTGCCGATGCGTGGATGGCGGTGGGTCACTGGAGAATACACGTCACAATGTGCTTGTGCGAGAAGTGGTGGGAATTCAAAACGAAAGCCTGTGTGGTGATCTGTGACCTACGCCGGCATTACATGCAAACATGGAGCCAAACACTATACATTTTCCCGACATGTATGTTCATGCACGAGGTAATCTGGCTGTGGCTAGTTCTCCGTTCCGAACCCTGGTACAATATGCCGGTTAGAATGGATTGGTTCCAAATGGATAGGATGAAGTGGAACAGCTATTTCCGTAAAATGTGCTGCGATCGCGCATATCACCACGAAATATCACCGGTTCAAAAAGACAAACCCTTTAGTTTGATGTACGACTATAATATTGGGTGCGACATGTGGACCCTGATGTTCAAAGCACAGGTTTCCAACGAGTGGAAGTATGATGCGTTCTACAGTCGGACTTTGGCCGGCACTATGGGTGAATTCT

#Run CUDA_RNA fold

Process in parallel < G/(4(n^2)) where G is GPU RAM in bytes and n is the length of the RNA sequences

In [ ]:
def truncate_fasta(fastapath, newlen):
  lines= []
  with open(fastapath, 'r') as f:
      lines = f.readlines()
  print("Old length of lines: ", len(lines))
  lines = lines[:newlen]
  print("New length of lines: ", len(lines))
  with open(fastapath, 'w') as f:
    for line in lines:
      f.write(line)

#truncate_fasta(dst, 100)

in_parallel = int(total_mem/(4*(length*length)))
print(in_parallel)
print("Multiplier to get to actual limit: ", 100/in_parallel)

188
Multiplier to get to actual limit:  0.5319148936170213


In [ ]:
#%%timeit
odest = "./RNA_list_out.fasta"
#%timeit -r 1 !RNA_CUDA_GRAPH=0 /home/ViennaRNA/bin/RNAfold -i {dst} -o {odest}
%timeit -r 1 !RNA_BACKTRACK_THREADS=auto /home/ViennaRNA/bin/RNAfold -i {dst} -o {odest}
#RNA_BACKTRACK_THREADS=auto

#!/home/ViennaRNA/bin/RNAfold -i {dst} -o {odest}
#!/home/ViennaRNA/bin/RNAfold -i {dst} -o {odest} - j  {in_parallel}

./modular_decomposition.cu init_gpu(100, 5400)
./modular_decomposition.cu fmli_kernel block size 768, modular_decomposition_kernel block size 768 (both were hardcoded 64)
./int_loop.cu            init_gpu2(100,VC,3,5400,512)
./hp_mb_loop.cu          init_gpu3(100,VC,3,5400,512)
./int_loop.cu            int_loop_kernel block size 64 (was hardcoded 32)
./hp_mb_loop.cu          hp_mb_3p_kernel block size 768 (was hardcoded 512)
./int_loop.cu            load_my_c_kernel block size 768 (was hardcoded 512)
./modular_decomposition.cu CUDA graph capture for load_fML/modular_decomposition/load_min_fML: enabled
./modular_decomposition.cu CUDA graph stats: 5394 update() succeeded, 1 first-time instantiate, 1 forced reinstantiate (update failed), 0.087 s cumulative capture/update/instantiate/destroy overhead (excludes launch+sync)
mfe_cuda.c               phase timing (s): int_loop=24.502 hp_mb=3.583 load_my_c=1.897 modular_decomp=56.574 | new_c_host=92.443 fml_host=81.412 my_fml_update_host=35.19

Test 01: At RNAs 400, J 180, and len 1500: 2min 23s ± 484 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)

Test 02: At RNAs 4, J --, and len 5601: 55.2 s ± 789 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)

Test 03: At RNAs 20 J --, and len 5601: 2min 16s ± 799 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)

Test 04: At RNAs 30 J --, and len 5601: 3min 39s ± 7.2 s per loop (mean ± std. dev. of 7 runs, 1 loop each)

Test 05: At RNAs 34 J--, and len 5601: 4min 4s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)

Test 06: At RNAs 100 J--, and len 5601: 11min 18s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each) on L4

Test 07: At RNAs 80 J--, and len 5601: 9min 3s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each) on L4 with CUDA_Graphs

Test 08: At RNAs 100 J--, and len 5601: 11min 41s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each) on L4 with CUDA_Graphs

Test 09: At RNAs 100 J--, and len 5601: 11min 15s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each) on L4 with CUDA_Graphs

Test 10: At RNAs 100 J--, and len 5601: 11min 34s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each) on L4 with CUDA_Graphs

Test 11: At RNAs 100 J--, and len 5601: 11min 20s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each) on L4 with CUDA_Graphs

Test 12: At RNAs 100 J--, and len 5601: 6min 41s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each) on L4 with CUDA_Graphs and GPU energy precompute port

Test 13: At RNAs 100 J--, and len 5601: 6min 39s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each) on L4 with CUDA_Graphs and GPU energy precompute port

Test 14: At RNAs 100 J--, and len 5601: 6min 31s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each) on L4 with Heterogenous_Computing_GPU_Energy_Precompute_CUDA_Graphs

Test 15: At RNAs 100 J--, and len 5601: 6min 21s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each) on L4 with Heterogenous_Computing_GPU_Energy_Precompute_CUDA_Graphs

Test 16: At RNAs 400 J--, and len 5601: 27min ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each) on L4 with Heterogenous_Computing_GPU_Energy_Precompute_CUDA_Graphs

Test 17 At RNAs 65505 J-- and len 400: 8min ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each) on L4 with Heterogenous_Computing_GPU_Energy_Precompute_CUDA_Graphs

Test 18 At RNAs 50000 J-- and len 400: 8min ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each) on L4 with Heterogenous_Computing_GPU_Energy_Precompute_CUDA_Graphs

Test 19 At RNAs 100000 J-- and len 400: 8min 13s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each) on L4 with Heterogenous_Computing_GPU_Energy_Precompute_CUDA_Graphs CUDA_GRAPHS DISABLED

Test 20 At RNAs 100000 J-- and len 400: 16min 27s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each) on L4 with Heterogenous_Computing_GPU_Energy_Precompute_CUDA_Graphs CUDA_GRAPHS ENABLED

Test 21 At RNAs 10 J-- and len 10000: 2min 8s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each) on L4 with Heterogenous_Computing_GPU_Energy_Precompute_CUDA_Graphs CUDA_GRAPHS ENABLED

Test 22 At RNAs 10 J-- and len 32000: 32min 54s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each) on L4 with Heterogenous_Computing_GPU_Energy_Precompute_CUDA_Graphs CUDA_GRAPHS ENABLED

Test 23 At RNAs 400 J-- and len 5601: 26min 57s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each) on L4 with Langdon_Indexing_Logic_Het_Energy_Precompute_CUDA_Graphs CUDA_GRAPHS ENABLED

Test 24 At RNAs 400 J-- and len 5601: 26min 59s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each) on L4 with Langdon_Indexing_Logic_Het_Energy_Precompute_CUDA_Graphs CUDA_GRAPHS ENABLED


*Pulled in Langdon's edits from main


Test 25 At RNAs 400 J-- and len 5601: 26min 25s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each) on L4 with Langdon_Indexing_Logic_Het_Energy_Precompute_CUDA_Graphs CUDA_GRAPHS ENABLED


Test 26 At RNAs 1000 J-- and len 1000: 2min 24s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each) on L4 with main's langdon commits. Cannot handle long RNAs


Test 26 At RNAs 1000 J-- and len 1000: 1min 32s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each) on L4 with Langdon_Indexing_Logic_Het_Energy_Precompute_CUDA_Graphs CUDA_GRAPHS ENABLED


Test 26 At RNAs 1000 J-- and len 1000: 1min 28s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each) on L4 with Heterogenous_Computing_GPU_Energy_Precompute_CUDA_Graphs CUDA_GRAPHS ENABLED


Test 27 At RNAs 400 J-- and len 1500: 1min 23s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each) on L4 with Heterogenous_Computing_GPU_Energy_Precompute_CUDA_Graphs CUDA_GRAPHS ENABLED


Test 27 At RNAs 400 J-- and len 1500: 1min 28s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each) on L4 with Langdon_Indexing_Logic_Het_Energy_Precompute_CUDA_Graphs CUDA_GRAPHS ENABLED


Test 27 At RNAs 400 J-- and len 1500: 1min 28s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each) on L4 with main's langdon commits



In [ ]:
dictt_list = [(4,55.2), (20,136), (30,219), (34,244), (100,678), (100,701), (100,675), (100,694), (100, 680), (100, 399), (100, 391), (100,381), (400, 1620)]
for key, value in dictt_list:
  print(value/key)

print("___________________")
print((60*8)/65505)
print("___________________")
print(((60*8) + 13)/100000)
print("___________________")

13.8
6.8
7.3
7.176470588235294
6.78
7.01
6.75
6.94
6.8
3.99
3.91
3.81
4.05
___________________
0.0073276849095488894
___________________
0.00493
___________________


#NCU (Nsight Compute) kernel profiling

Runs a small dedicated profiling input through NCU, filtered to this project's own GPU kernels (across modular_decomposition.cu / int_loop.cu / hp_mb_loop.cu), and writes both a raw CSV and a summarized text file so results can be handed back for analysis without needing the Nsight Compute GUI.

In [ ]:
%%bash
# Nsight Compute (ncu) ships alongside nvcc in the CUDA toolkit on most Colab
# GPU images, but it's not guaranteed -- check before assuming it's there.
if command -v ncu >/dev/null 2>&1; then
  echo "ncu found: $(command -v ncu)"
  ncu --version
else
  echo "ncu not on PATH -- attempting apt install of nsight-compute"
  apt-get -qq update
  apt-get -qq install -y nsight-compute 2>&1 | tail -20 || \
    echo "apt install failed -- ncu may need a manual .deb/.run install from NVIDIA's site; report this back before running the profiling cell below"
fi

ncu found: /usr/local/cuda/bin/ncu
NVIDIA (R) Nsight Compute Command Line Profiler
Copyright (c) 2018-2025 NVIDIA Corporation
Version 2025.1.1.0 (build 35528883) (public-release)


In [ ]:
# Separate, deliberately small input for NCU profiling -- NCU instruments
# and replays every kernel launch it captures, and RNAfold's DP loop issues
# roughly half a dozen kernel launches per outer i iteration (O(length)
# iterations), so profiling anything close to the stress_400/RNA_list-sized
# runs above would multiply wall time by 10-100x for no extra insight into
# *which* kernel is the bottleneck.
#
# ncu_num_rnas is derived from the actual device SM count (not hardcoded --
# the earlier nfiles=8-on-a-58-SM-GPU profiling run showed everything as
# falsely "latency-bound" purely because the batch was narrower than the
# device, so this must track whatever GPU Colab actually assigns, not a
# number copied from a previous session).
#
# ncu_length was 300 (~296 outer-loop rows); widened to 900 to give
# int_loop_kernel's BLOCK_SIZE candidates a longer tail of large-grid rows
# to run against, not just the first few. Note this does NOT fix the
# BLOCK_SIZE auto-tuner's own first-call-only sampling bug (still open,
# code-side) -- it just gives us more post-benchmark launches to inspect.
ncu_num_rnas = torch.cuda.get_device_properties(0).multi_processor_count
ncu_length   = 900  # divisible by 3 -- generate_random_codon_perms requires this
ncu_fasta    = "./RNA_list_ncu.fasta"
ncu_out      = "./RNA_list_ncu_out.fasta"

ncu_rna_list = generate_random_codon_perms(ncu_length, ncu_num_rnas)
make_in_fasta(ncu_rna_list, ncu_fasta)
print(f"Wrote {ncu_num_rnas} sequences (= device SM count) of length {ncu_length} to {ncu_fasta}")

In [ ]:
# --set basic: SpeedOfLight + a handful of top-level metrics -- fast enough
#   to run per-launch across hundreds of launches. Swap to --set full (much
#   slower) only once ncu_summary.txt below has pointed at one kernel worth
#   drilling into, and add --kernel-id to restrict to just that kernel.
# --kernel-name regex: restrict to RNAfold's own __global__ kernels (the
#   complete list, across modular_decomposition.cu/int_loop.cu/hp_mb_loop.cu)
#   so NCU doesn't also spend time profiling unrelated CUDA-runtime kernels.
# RNA_CUDA_GRAPH=0: profile kernels individually rather than as opaque graph
#   nodes, so per-kernel attribution in the CSV stays simple. Re-test with
#   graphs on afterward if graph capture overhead itself becomes a question.
ncu_kernels = (
    "init_fML_kernel|load_fML_kernel|load_min_fML_kernel|fmli_kernel|"
    "modular_decomposition_kernel|init_my_c_kernel|load_my_c_kernel|"
    "int_loop_kernel|hp_mb_3p_kernel"
)

!RNA_CUDA_GRAPH=0 ncu \
  --set basic \
  --kernel-name "regex:{ncu_kernels}" \
  --csv \
  --log-file ncu_report.csv \
  -- /home/ViennaRNA/bin/RNAfold -i {ncu_fasta} -o {ncu_out}

# If this errors out with ERR_NVGPUCTRPERM: GPU performance counters are
# restricted and ncu needs to run as root/with elevated privilege -- retry
# the same command with a leading `sudo` (Colab's shell is already root by
# default, so this is usually NOT needed, but some GPU types/driver configs
# on Colab do lock counters down).
!wc -l ncu_report.csv && head -3 ncu_report.csv

./modular_decomposition.cu init_gpu(58, 300)
./modular_decomposition.cu fmli_kernel block size 768, modular_decomposition_kernel block size 768 (both were hardcoded 64)
./int_loop.cu            init_gpu2(58,VC,3,300,512)
./hp_mb_loop.cu          init_gpu3(58,VC,3,300,512)
./int_loop.cu            int_loop_kernel candidate BLOCK_SIZE=32 : 259.3587 ms (best of 3)
./int_loop.cu            int_loop_kernel candidate BLOCK_SIZE=64 : 261.0872 ms (best of 3)
./int_loop.cu            int_loop_kernel candidate BLOCK_SIZE=128: 256.4188 ms (best of 3)
./int_loop.cu            int_loop_kernel candidate BLOCK_SIZE=256: 251.6337 ms (best of 3)
./int_loop.cu            int_loop_kernel block size 256 chosen by timed benchmark (was hardcoded 32)
./hp_mb_loop.cu          hp_mb_3p_kernel block size 768 (was hardcoded 512)
./int_loop.cu            load_my_c_kernel block size 768 (was hardcoded 512)
./modular_decomposition.cu CUDA graph capture for load_fML/modular_decomposition/load_min_fML: disabled (RNA_

In [ ]:
import pandas as pd
from io import StringIO

# ncu's --log-file writes its own "==PROF== Connected/Disconnected" status
# chatter into the same file as the CSV report (it's not a clean CSV on its
# own), so find the real header row rather than assuming line 1 is it.
with open("ncu_report.csv") as f:
    raw_lines = f.readlines()
header_idx = next(i for i, l in enumerate(raw_lines) if l.startswith('"ID"'))
if header_idx > 0:
    print(f"Dropping {header_idx} non-CSV line(s) ncu wrote before the header:")
    for l in raw_lines[:header_idx]:
        print("  ", l.rstrip())

df = pd.read_csv(StringIO("".join(raw_lines[header_idx:])))
print("Columns:", list(df.columns))
print(df.head())

# ncu's --csv layout differs a bit by version -- this is a best-effort
# per-kernel summary. If the column names below don't match what's printed
# above, that printout (plus the raw ncu_report.csv) is what to hand back
# rather than trusting this block, since it's guessing at column names it
# hasn't been run against.
summary_lines = []
try:
    kernel_col = "Kernel Name" if "Kernel Name" in df.columns else next(
        c for c in df.columns if "kernel" in c.lower() and "name" in c.lower())
    if "Metric Name" in df.columns and "Metric Value" in df.columns:
        # long format: one row per (kernel launch, metric)
        wanted = ["Duration", "Compute (SM) Throughput", "Memory Throughput",
                  "DRAM Throughput", "Achieved Occupancy"]
        sub = df[df["Metric Name"].isin(wanted)].copy()
        sub["Metric Value"] = pd.to_numeric(sub["Metric Value"], errors="coerce")
        pivot = sub.groupby([kernel_col, "Metric Name"])["Metric Value"].agg(["count", "mean", "sum"])
        summary_lines.append(pivot.to_string())
    else:
        # wide format: metrics are already columns
        counts = df.groupby(kernel_col).size().rename("launch_count")
        summary_lines.append(counts.to_string())
        num_cols = df.select_dtypes("number").columns
        summary_lines.append(df.groupby(kernel_col)[num_cols].mean().to_string())
except Exception as e:
    summary_lines.append(f"Auto-summary failed: {e!r} -- see raw ncu_report.csv / df.columns above instead")

with open("ncu_summary.txt", "w") as f:
    f.write("\n\n".join(summary_lines))

print("\n".join(summary_lines))

Dropping 2 non-CSV line(s) ncu wrote before the header:
   ==PROF== Connected to process 27218 (/home/ViennaRNA/bin/RNAfold)
   ==PROF== Disconnected from process 27218
Columns: ['ID', 'Process ID', 'Process Name', 'Host Name', 'Kernel Name', 'Context', 'Stream', 'Block Size', 'Grid Size', 'Device', 'CC', 'Section Name', 'Metric Name', 'Metric Unit', 'Metric Value', 'Rule Name', 'Rule Type', 'Rule Description', 'Estimated Speedup Type', 'Estimated Speedup']
   ID  Process ID Process Name  Host Name  \
0   0       27218      RNAfold  127.0.0.1   
1   0       27218      RNAfold  127.0.0.1   
2   0       27218      RNAfold  127.0.0.1   
3   0       27218      RNAfold  127.0.0.1   
4   0       27218      RNAfold  127.0.0.1   

                              Kernel Name  Context  Stream   Block Size  \
0  init_my_c_kernel(unsigned long, int *)        1       7  (512, 1, 1)   
1  init_my_c_kernel(unsigned long, int *)        1       7  (512, 1, 1)   
2  init_my_c_kernel(unsigned long, int *) 

In [ ]:
from google.colab import files
files.download("ncu_report.csv")
files.download("ncu_summary.txt")
# Uncomment to also grab the full binary report (opens in the Nsight Compute
# GUI locally for a visual timeline/section drill-down) -- much bigger file,
# only worth it once ncu_summary.txt has pointed at a specific kernel:
# files.download("ncu_report.ncu-rep")

#Run normal RNAFold

In [ ]:
def foldallRNAs(rnalist, tot = 1):
  results = []
  num = 1
  for seq in rnalist[:tot]:
    num += 1
    mfe, struc = RNA.fold(seq)
    results.append((mfe, struc))
  return results
num_read = 5
#%timeit -r 1 foldallRNAs(RNA_list)
results = foldallRNAs(RNA_list, num_read)
for r in results:
  print(r)

At RNAs 34, J --, and len 5601:

In [ ]:
for struc, mfe in results:
  print(mfe)
  print(struc)
  print("%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%")

#Compare the outputs

In [ ]:
#Parse and extract the data from fasta
def parse_fold_file(path):
    records = []
    header = seq = None
    with open(path, 'r') as f:
        for raw in f:
            line = raw.rstrip('\n')
            if not line:
                continue  # defensive -- skip any stray blank lines
            if line.startswith('>'):
                header = line[1:]
                seq = None
            elif seq is None:
                seq = line
            else:
                records.append((header, seq, line))  # (header, sequence, "structure (energy)")
                header = seq = None
    return records

#Read the results from odest and from results and compare
def compare_results(odest, results, num_to_compare = None):
  if num_to_compare == None:
    num_to_compare = len(results)

  odest_results = parse_fold_file(odest)
  print("^^^^^^^^^^^^^^^^")
  #with open(odest+'_RNA.fold', 'r') as f:
  #  for n in range(0, 1000):
  #    print(f.readline(n))
  #print("^^^^^^^^^^^^^^^^")
  for i in range(0, num_to_compare):
    a = results[i][0]
    b = odest_results[i][2].split(' ')[0]
    #print(a)
    #print(b)
    if a == b:
      print("Match")
    else:
      print("No Match")
    #print("%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%")
  print("Done")


compare_results(odest + '_RNA.fold', results, 5)
